<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.5-lora-and-qlora-adapters/notebooks/exercises-7.5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.5 — LoRA & QLoRA Adapters Deep Dive
Netsetos GenAI Course v2.0 | Module 7

Rank ablation, target module ablation, PEFT variants, QLoRA internals, adapter lifecycle, India patterns


In [ ]:
print('Module 7: Fine-Tuning LLMs')
print('Lesson 7.5: LoRA & QLoRA Adapters — Advanced Deep Dive')


## Ex 1: Rank Ablation Simulation


In [ ]:
import random
random.seed(42)

def simulate_rank_ablation(rank, lr_tuned=False):
    base_quality = 75.0
    rank_gain = min(rank * 0.15, 12)  # diminishing returns
    lr_penalty = 0 if lr_tuned else (rank / 16) * 3  # wrong LR hurts more at high rank
    noise = random.gauss(0, 0.5)
    return base_quality + rank_gain - lr_penalty + noise

print('Rank ablation — fixed LR vs tuned LR:')
print(f'{"Rank":>6s} {"Fixed LR":>10s} {"Tuned LR":>10s} {"Difference":>12s}')
for r in [4, 8, 16, 32, 64, 128, 256]:
    fixed = simulate_rank_ablation(r, lr_tuned=False)
    tuned = simulate_rank_ablation(r, lr_tuned=True)
    print(f'{r:>6d} {fixed:>10.1f}% {tuned:>10.1f}% {tuned-fixed:>+10.1f}pp')
print()
print('Key insight: At high rank, fixed LR DEGRADES quality!')
print('LR tuning matters more than rank choice.')


## Ex 2: Target Module Impact


In [ ]:
print('Target module impact (relative to full FT = 100%):')
print()
for config, quality, params in [
    ('q_proj + v_proj only', 82, '4.19M'),
    ('All attention (q/k/v/o)', 87, '8.39M'),
    ('MLP only (gate/up/down)', 91, '12.58M'),
    ('All linear (attention+MLP)', 96, '20.28M'),
    ('All linear + embed + LM head', 94, '28.67M'),  # WORSE without decoupled LR!
]:
    print(f'  {config:35s}: {quality}% quality | {params} params')
print()
print('Key: MLP-only > attention-only at same param count')
print('Including embed/LM head WITHOUT decoupled LR hurts!')


## Ex 3: PEFT Variant Comparison


In [ ]:
print('PEFT Variant Benchmarks (Llama 7B, GSM8K):')
print()
for variant, accuracy, flag in [
    ('Vanilla LoRA', 67.7, 'baseline'),
    ('+ DoRA', 72.1, 'use_dora=True'),
    ('+ rsLoRA', 68.9, 'use_rslora=True'),
    ('+ PiSSA init', 72.9, 'init_lora_weights="pissa"'),
    ('DoRA + rsLoRA + PiSSA', 73.4, 'all three'),
    ('Full fine-tuning', 71.2, 'reference'),
]: print(f'  {variant:25s}: {accuracy}% | {flag}')
print()
print('Note: DoRA+PiSSA can BEAT full fine-tuning!')
print('BUT: PiSSA fails catastrophically in GRPO — use vanilla for RL')


## Ex 4: NF4 vs INT4 vs FP4


In [ ]:
print('Quantization format comparison:')
print()
for fmt, description, quality in [
    ('NF4', 'Non-uniform bins at N(0,1) quantiles', 'Best'),
    ('FP4', '4-bit floating point (2 exp + 1 mantissa)', 'Good'),
    ('INT4', 'Uniform 16 levels across range', 'Worst'),
]: print(f'  {fmt:5s}: {description:45s} → {quality}')
print()
print('Why NF4 wins:')
print('  • LLM weights are approximately normally distributed')
print('  • NF4 bins: more near zero (where weights cluster)')
print('  • INT4 bins: uniformly spaced (wastes bits in tails)')
print('  • FP4 bins: more near zero BUT less precise than NF4')
print()
print('Double quantization saves ~3GB on 65B model')
print('  Quantizes FP32 absmax constants → FP8 (0.37 bits/param)')


## Ex 5: Adapter Merging


In [ ]:
print('Adapter merging techniques:')
print()
for method, density, description in [
    ('Linear', 'N/A', 'Weighted sum of task vectors'),
    ('TIES', '0.5', 'Trim small → majority sign → disjoint merge'),
    ('DARE', '0.7-0.8', 'Random drop 90-99% → rescale remainder'),
    ('DARE-TIES', '0.7', 'Best of both: 73.46 avg on benchmarks'),
]: print(f'  {method:12s} (density={density:8s}): {description}')
print()
print('Decision matrix:')
print('  Merge → single-task deploy, zero overhead')
print('  Multi-LoRA → multi-task, per-request routing')
print('  Stack → compositional capabilities (experimental)')


## Ex 6: Multi-LoRA Serving Architecture


In [ ]:
print('Multi-LoRA serving stack:')
print()
for system, throughput, feature in [
    ('Naive (separate models)', '1×', 'N models × full VRAM'),
    ('vLLM multi-LoRA', '~50-80% of base', 'OpenAI API compatible'),
    ('S-LoRA (MLSys 2024)', '4× over naive vLLM', 'Unified paging, 1000s adapters'),
    ('Punica kernel', '12× over SOTA', '2ms per-token overhead'),
    ('dLoRA', '1.8× over S-LoRA', 'Dynamic merged/unmerged switching'),
]: print(f'  {system:25s}: {throughput:20s} | {feature}')
print()
print('Memory comparison (7B model, 3 adapters):')
print('  3 separate models: 42 GB')
print('  Multi-LoRA: 14 GB + 150 MB = 14.15 GB (3× savings)')


## Ex 7: Compute Dtype Selection


In [ ]:
print('Compute dtype decision:')
print()
for task, dtype, reason in [
    ('SFT (instruction tuning)', 'bfloat16', '8 exponent bits = float32 range, no overflow'),
    ('DPO (preference tuning)', 'float16', 'Smaller gradient norms, more stable'),
    ('GRPO (reasoning RL)', 'float16', 'Unsloth: "dramatically better for RL"'),
    ('Continued pretraining', 'bfloat16', 'Long sequences benefit from bf16 stability'),
]: print(f'  {task:30s}: {dtype:10s} | {reason}')
print()
print('Gradient checkpointing: 50-70% activation memory reduction')
print('  Cost: 20-33% slower training')
print('  ALWAYS: gradient_checkpointing_kwargs={"use_reentrant": False}')


## Ex 8: India Adapter Ecosystem


In [ ]:
print('India LoRA adapter ecosystem:')
print()
print('Proven Airavata config (AI4Bharat):')
for k,v in [('Base model','OpenHathi-7B (Llama 2)'),('Rank','16'),('Alpha','32'),
    ('Dropout','0.05'),('Targets','q/k/v/gate/up/down_proj'),
    ('LR','5e-4'),('Epochs','4'),('Key finding','LoRA preserves English; full FT degrades it')]:
    print(f'  {k:15s}: {v}')
print()
print('Community adapters on HuggingFace:')
for name, desc in [('ai4bharat/Airavata','Hindi instruction-tuned, 7B'),
    ('Tamil-Llama','Bilingual Tamil+English, 16K new tokens'),
    ('Sarvam-Translate','Gemma3-4B, 22 Indian languages'),
    ('Navarasa 2.0','Gemma-7B, 15 Indic languages, 650K samples'),
    ('PHI-4-Hindi-LoRA','MMLU Pro 52.39'),
]: print(f'  {name:25s}: {desc}')
print()
print('Multilingual insight: language-specific LoRA > joint training')
print('  for low-resource Indic languages')


---
## Recap
The hierarchy of LoRA impact: LR tuning (most important, ~10× higher than full FT) > target modules (all-linear, MLP > attention) > rank (least important, r=16-64 for most tasks). All LoRA variants converge within 0.43% when LR is swept. DoRA (+1-4%) and rsLoRA (stable high rank) are one-line upgrades. PiSSA (+5% GSM8K) works for SFT but fails in GRPO. NF4 is information-optimal for normally distributed weights. bf16 for SFT, fp16 for RL. Multi-LoRA serving (S-LoRA, Punica) enables 1000s of concurrent adapters with 2ms overhead. DARE-TIES merging for combining capabilities. For India: LoRA preserves English while adding Indic languages (Airavata), language-specific adapters outperform joint training, and community adapters exist for Hindi, Tamil, Telugu, and 15+ languages.
